> **Runs automatically.** This copy lives in the `football-picks` repository on GitHub, which runs it on a schedule and keeps the logs in the repository's `data` folder. Open it here to read or change it; the schedule picks up whatever is saved in the repository.

# Post-game review v1: what the models got right and wrong, and why

This notebook reads your pick logs, `nfl_pick_log.csv` and `ncaa_pick_log.csv`, and reviews every game the models picked once it's been played. For each game, it pulls both teams' stats from the play-by-play, works out how much of the model's miss came from each part of the game, and writes a short summary in plain language.

The reviews are saved next to the pick logs, as `nfl_game_reviews.csv` and `ncaa_game_reviews.csv`. They're built for the app: a "how did we do" screen, and the facts an AI needs to explain each game without making anything up.

**What it does**
1. Loads the pick logs. When more than one version of a model picked a game, it reviews the newest pick.
2. Pulls each team's stats from the play-by-play: plays, successful plays, big plays, turnovers, defensive and return touchdowns, and finishing drives.
3. Learns from last season what each of those is worth in points.
4. Splits every miss into parts and writes a summary of each game.
5. Shows the latest week's biggest misses.
6. Shows the big picture: how often the model was close, and why it missed.
7. Shows which teams have surprised the model most this season.

**To run it:** in Colab, choose **Runtime → Run all**. It takes a minute or two, and the first run a little longer. Allow access to your Google Drive when Colab asks. It only reads the pick logs; it never changes them or the models, and it doesn't use any CollegeFootballData calls.

**Every week:** run it on Tuesday, after the NFL and college model notebooks.

In [ ]:
%pip install -q nflreadpy "polars>=1.0"

## Step 1: Load the pick logs

For each game, the review uses the main margin model's pick (not Elo's) and the totals model's pick, along with the market lines saved with them. If a league's log doesn't exist yet, that league is skipped.

In [ ]:
import io
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import polars as pl
import requests
import nflreadpy as nfl

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(400)
now_utc = datetime.now(timezone.utc)

# The pick logs and the review files live in your Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
except ImportError:
    DRIVE = Path(".")
CACHE = DRIVE / "game_review"          # play-by-play summaries of finished seasons, saved so they're downloaded once

LEAGUES = ["NFL", "College"]
LOGS = {"NFL": DRIVE / "nfl_pick_log.csv", "College": DRIVE / "ncaa_pick_log.csv"}
REVIEW_FILES = {"NFL": DRIVE / "nfl_game_reviews.csv", "College": DRIVE / "ncaa_game_reviews.csv"}
ELO_PREFIX = {"NFL": "elo_v", "College": "cfb_elo_v"}       # Elo picks are left out; the review uses the main models
NUMBERS = ["model_p_home", "market_p_home", "model_spread", "market_spread", "model_total", "market_total", "model_p_over"]


def read_log(league):
    """The pick log as text, with the numbers and game ids converted."""
    log = pl.read_csv(LOGS[league], infer_schema_length=0)
    log = log.with_columns(pl.lit(None, pl.String).alias(c) for c in NUMBERS if c not in log.columns)
    return log.with_columns(
        *[pl.col(c).cast(pl.Float64, strict=False) for c in NUMBERS],
        pl.col("season").cast(pl.Int32), pl.col("week").cast(pl.Int32),
        pl.col("game_id").cast(pl.Int64 if league == "College" else pl.String),
    )


def picks_by_game(log, league):
    """One row per game: the newest margin pick (if more than one version picked it) and the totals pick."""
    games = log.sort("logged_at_et").group_by("game_id").agg(
        pl.col("season").last(), pl.col("week").last(), pl.col("kickoff_et").last(),
        pl.col("away_team").last(), pl.col("home_team").last())
    margin = (log.filter(pl.col("model_spread").is_not_null() & ~pl.col("model").str.starts_with(ELO_PREFIX[league]))
              .sort("logged_at_et").group_by("game_id")
              .agg(margin_model=pl.col("model").last(), model_spread=pl.col("model_spread").last(), model_p_home=pl.col("model_p_home").last(),
                   market_spread=pl.col("market_spread").last(), market_p_home=pl.col("market_p_home").last()))
    totals = (log.filter(pl.col("model_total").is_not_null()).sort("logged_at_et").group_by("game_id")
              .agg(totals_model=pl.col("model").last(), model_total=pl.col("model_total").last(),
                   market_total=pl.col("market_total").last(), model_p_over=pl.col("model_p_over").last()))
    return games.join(margin, on="game_id", how="left").join(totals, on="game_id", how="left").filter(
        pl.col("model_spread").is_not_null() | pl.col("model_total").is_not_null())


picks = {}
for league in LEAGUES:
    if not LOGS[league].exists():
        print(f"{league}: no pick log yet at {LOGS[league]}. Run the {league} model notebook first, and its picks will be reviewed here.")
        continue
    picks[league] = picks_by_game(read_log(league), league)
    p = picks[league]
    print(f"{league}: {p.height} games picked, seasons {p['season'].min()}–{p['season'].max()}, "
          f"from models {', '.join(sorted(set(p['margin_model'].drop_nulls().to_list() + p['totals_model'].drop_nulls().to_list())))}.")

## Step 2: Game stats from the play-by-play

For each team in each game:

| Stat | What it counts |
|---|---|
| `plays`, `successes` | Runs and passes, and how many were successful: half the yards to go on 1st down, 70% of them on 2nd, all of them on 3rd or 4th. It's the same success rate the college model uses |
| `explosive` | Big plays: runs and passes of 20+ yards |
| `turnovers` | Interceptions and lost fumbles, including muffed punts and kickoffs |
| `return_tds` | Touchdowns scored on defense or special teams |
| `trips`, `trip_points` | Drives that reached the opponent's 40, and the points scored on them: 7 for a touchdown, 3 for a field goal |

College play-by-play comes from the same sportsdataverse files as the college model, and NFL play-by-play from nflverse. Finished seasons are saved in a `game_review` folder in your Drive, so they're only downloaded once. A game is reviewed once both teams' play-by-play is in, which usually takes a day or two after the game.

In [ ]:
CFB_DATA = "https://raw.githubusercontent.com/sportsdataverse/cfbfastR-data/main"
CFB_PBP = "https://github.com/sportsdataverse/sportsdataverse-data/releases/download/cfbfastR_cfb_pbp/play_by_play_{}.parquet"
CFB_COLUMNS = ["game_id", "pos_team", "home_team", "home_team_id", "away_team_id", "rush", "pass", "down", "distance",
               "yards_gained", "EPA", "play_type", "defense_score_play", "yards_to_goal", "drive_id"]
NFL_COLUMNS = ["game_id", "posteam", "play_type", "down", "ydstogo", "yards_gained", "interception", "fumble_lost", "fumbled_1_team",
               "touchdown", "pass_touchdown", "rush_touchdown", "td_team", "yardline_100", "fixed_drive", "fixed_drive_result"]
STATS = ["plays", "successes", "yards", "explosive", "turnovers", "return_tds", "trips", "trip_points"]
MIN_PLAYS = 30                       # games where either team has fewer runs and passes in the data are left out
CURRENT_SEASON = now_utc.year if now_utc.month >= 7 else now_utc.year - 1


def needed_yards(down, distance):
    """A play is successful if it gains half the yards to go on 1st down, 70% of them on 2nd, and all of them on 3rd or 4th."""
    return pl.when(down == 1).then(0.5).when(down == 2).then(0.7).otherwise(1.0) * distance


def finish(off, give, rtd, trips):
    out = (off.join(give, on=["game_id", "team"], how="left").join(rtd, on=["game_id", "team"], how="left")
           .join(trips, on=["game_id", "team"], how="left"))
    return out.with_columns(pl.col(c).fill_null(0).cast(pl.Float64) for c in STATS)


def cfb_team_stats(p):
    """College: one row per team per game, from the play-by-play."""
    p = p.with_columns(
        game_id=pl.col("game_id").cast(pl.Int64),
        team=pl.when(pl.col("pos_team") == pl.col("home_team")).then(pl.col("home_team_id")).otherwise(pl.col("away_team_id")).cast(pl.Int64),
        other=pl.when(pl.col("pos_team") == pl.col("home_team")).then(pl.col("away_team_id")).otherwise(pl.col("home_team_id")).cast(pl.Int64),
        scrimmage=((pl.col("rush") == 1) | (pl.col("pass") == 1)) & pl.col("EPA").is_not_null(),
    )
    s = p.filter("scrimmage").with_columns(success=(pl.col("yards_gained") >= needed_yards(pl.col("down"), pl.col("distance"))).fill_null(False))
    off = s.group_by("game_id", "team").agg(plays=pl.len(), successes=pl.col("success").sum(), yards=pl.col("yards_gained").sum(),
                                            explosive=(pl.col("yards_gained") >= 20).sum())
    give = pl.concat([   # interceptions and lost fumbles by the offense, and muffed kicks by the return team
        p.filter(pl.col("play_type").is_in(["Interception Return", "Interception Return Touchdown", "Fumble Recovery (Opponent)",
                                            "Fumble Recovery (Opponent) Touchdown", "Fumble Return Touchdown"])).select("game_id", "team"),
        p.filter(pl.col("play_type").is_in(["Punt Team Fumble Recovery", "Kickoff Team Fumble Recovery"])).select("game_id", team="other"),
    ]).group_by("game_id", "team").agg(turnovers=pl.len())
    rtd = p.filter((pl.col("defense_score_play") == 1) & pl.col("play_type").str.contains("Touchdown")).group_by(
        "game_id", team="other").agg(return_tds=pl.len())
    drive_points = p.filter(pl.col("play_type").is_in(["Rushing Touchdown", "Passing Touchdown", "Field Goal Good"])).group_by("game_id", "drive_id").agg(
        points=pl.when(pl.col("play_type") == "Field Goal Good").then(3).otherwise(7).max())
    trips = (s.filter(pl.col("yards_to_goal") <= 40).select("game_id", "drive_id", "team").unique()
             .join(drive_points, on=["game_id", "drive_id"], how="left")
             .group_by("game_id", "team").agg(trips=pl.len(), trip_points=pl.col("points").fill_null(0).sum()))
    return finish(off, give, rtd, trips)


def nfl_team_stats(p):
    """NFL: one row per team per game, from the play-by-play."""
    s = p.filter(pl.col("play_type").is_in(["pass", "run"]) & pl.col("down").is_not_null() & pl.col("posteam").is_not_null())
    s = s.with_columns(success=(pl.col("yards_gained") >= needed_yards(pl.col("down"), pl.col("ydstogo"))).fill_null(False))
    off = s.group_by("game_id", team="posteam").agg(plays=pl.len(), successes=pl.col("success").sum(), yards=pl.col("yards_gained").sum(),
                                                    explosive=(pl.col("yards_gained") >= 20).sum())
    give = pl.concat([
        p.filter(pl.col("interception") == 1).select("game_id", team="posteam"),
        p.filter(pl.col("fumble_lost") == 1).select("game_id", team="fumbled_1_team"),
    ]).drop_nulls().group_by("game_id", "team").agg(turnovers=pl.len())
    rtd = (p.filter((pl.col("touchdown") == 1) & (pl.col("pass_touchdown") == 0) & (pl.col("rush_touchdown") == 0)).drop_nulls("td_team")
           .group_by("game_id", team="td_team").agg(return_tds=pl.len()))
    trips = (s.filter(pl.col("yardline_100") <= 40).group_by("game_id", "fixed_drive", team="posteam").agg(result=pl.col("fixed_drive_result").first())
             .group_by("game_id", "team").agg(trips=pl.len(), trip_points=pl.when(pl.col("result") == "Touchdown").then(7)
                                                .when(pl.col("result") == "Field goal").then(3).otherwise(0).sum()))
    return finish(off, give, rtd, trips)


def fetch(url):
    for attempt in range(3):
        try:
            response = requests.get(url, timeout=300)
            if response.status_code == 404:
                return None
            response.raise_for_status()
            return response.content
        except requests.RequestException:
            time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"couldn't download {url.split('/')[-1]}")


def season_stats(league, season):
    """Team stats for every game in one season. Finished seasons are saved to your Drive after the first download."""
    finished = season < CURRENT_SEASON
    cache = CACHE / f"{'nfl' if league == 'NFL' else 'cfb'}_stats_{season}.parquet"
    if finished and cache.exists():
        return pl.read_parquet(cache)
    if league == "College":
        content = fetch(CFB_PBP.format(season))
        if content is None:
            return None
        stats = cfb_team_stats(pl.read_parquet(io.BytesIO(content), columns=CFB_COLUMNS))
    else:
        pbp = nfl.load_pbp([season])
        if pbp.is_empty():
            return None
        stats = nfl_team_stats(pbp.select(NFL_COLUMNS))
    if finished:
        CACHE.mkdir(parents=True, exist_ok=True)
        stats.write_parquet(cache)
    return stats


def final_scores(league, seasons):
    """Every finished game in these seasons, with home and away teams as they appear in the stats."""
    if league == "NFL":
        s = nfl.load_schedules(seasons).filter(pl.col("home_score").is_not_null())
        return s.select("game_id", "season", "week", home_key="home_team", away_key="away_team",
                        home_points=pl.col("home_score").cast(pl.Float64), away_points=pl.col("away_score").cast(pl.Float64))
    frames = []
    for season in seasons:
        s = pl.read_parquet(f"{CFB_DATA}/schedules/parquet/cfb_schedules_{season}.parquet")
        frames.append(s.select(pl.col("game_id").cast(pl.Int64), pl.col("season").cast(pl.Int32), pl.col("week").cast(pl.Int32),
                               home_key=pl.col("home_id").cast(pl.Int64), away_key=pl.col("away_id").cast(pl.Int64),
                               home_points=pl.col("home_points").cast(pl.Float64), away_points=pl.col("away_points").cast(pl.Float64)))
    return pl.concat(frames).filter(pl.col("home_points").is_not_null())


def game_table(league, seasons):
    """Finished games with both teams' stats; `has_stats` is False when the play-by-play isn't in yet."""
    scores = final_scores(league, seasons)
    stats = [s.with_columns(pl.col("team").cast(scores["home_key"].dtype)) for s in (season_stats(league, x) for x in seasons) if s is not None]
    if not stats:
        return scores.with_columns(has_stats=pl.lit(False))
    stats = pl.concat(stats).unique(["game_id", "team"])
    home = stats.select("game_id", home_key="team", **{f"h_{c}": c for c in STATS})
    away = stats.select("game_id", away_key="team", **{f"a_{c}": c for c in STATS})
    g = scores.with_columns(pl.col("game_id").cast(stats["game_id"].dtype)).join(home, on=["game_id", "home_key"], how="left").join(
        away, on=["game_id", "away_key"], how="left")
    return g.with_columns(has_stats=(pl.col("h_plays") >= MIN_PLAYS).fill_null(False) & (pl.col("a_plays") >= MIN_PLAYS).fill_null(False))


games_by_league = {}
for league, p in picks.items():
    seasons = sorted(set(p["season"].to_list()) | {s - 1 for s in p["season"].to_list()})
    t = time.time()
    games_by_league[league] = game_table(league, seasons)
    g = games_by_league[league]
    print(f"{league}: stats for {g.filter('has_stats').height:,} finished games in {seasons[0]}–{seasons[-1]} ({time.time() - t:.0f} seconds).")

## Step 3: What each part of a game is worth

From the previous season's games, the notebook learns two things:
- **Deserved points:** how many points usually go with a team's plays and successful plays. It's the same idea as in college model v3.
- **What else moves the score:** how many points of margin each turnover, defensive or return touchdown, extra big play and point of above-average finishing adds, beyond what success rate already counts.

Each season's reviews use what was learned from the season before.

In [ ]:
PARTS = ["efficiency", "turnovers", "return_tds", "big_plays", "finishing", "other"]


def lstsq(X, y):
    weights, *_ = np.linalg.lstsq(X, np.asarray(y, dtype=float), rcond=None)
    return weights


def deserved_points(g, weights, side):
    """Points a team usually scores with its number of plays and successful plays."""
    s, n = g[f"{side}_successes"].to_numpy(), g[f"{side}_plays"].to_numpy()
    return np.column_stack([s, n, s / n, np.ones(len(s))]) @ weights


def explainers(g, per_trip, total=False):
    """What can explain the gap between the deserved score and the real one: turnovers, return and defensive
    touchdowns, big plays (20+ yards), and finishing drives (points on trips inside the 40, compared with average)."""
    fin_h = g["h_trip_points"].to_numpy() - per_trip * g["h_trips"].to_numpy()
    fin_a = g["a_trip_points"].to_numpy() - per_trip * g["a_trips"].to_numpy()
    h, a = (lambda c: g[f"h_{c}"].to_numpy()), (lambda c: g[f"a_{c}"].to_numpy())
    if total:
        cols = [h("turnovers") + a("turnovers"), h("return_tds") + a("return_tds"), h("explosive") + a("explosive"), fin_h + fin_a]
    else:   # from the home team's side: takeaways, its return TDs, its big plays and its finishing, minus the other team's
        cols = [a("turnovers") - h("turnovers"), h("return_tds") - a("return_tds"), h("explosive") - a("explosive"), fin_h - fin_a]
    return np.column_stack(cols + [np.ones(g.height)])


def fit_season(g):
    """Learn from one finished season what each kind of play is worth in points."""
    g = g.filter("has_stats")
    team = pl.concat([g.select(s="h_successes", n="h_plays", pts="home_points"), g.select(s="a_successes", n="a_plays", pts="away_points")])
    X = np.column_stack([team["s"].to_numpy(), team["n"].to_numpy(), (team["s"] / team["n"]).to_numpy(), np.ones(team.height)])
    deserved = lstsq(X, team["pts"].to_numpy())
    per_trip = float((g["h_trip_points"].sum() + g["a_trip_points"].sum()) / (g["h_trips"].sum() + g["a_trips"].sum()))
    dh, da = deserved_points(g, deserved, "h"), deserved_points(g, deserved, "a")
    margin = lstsq(explainers(g, per_trip), (g["home_points"] - g["away_points"]).to_numpy() - (dh - da))
    total = lstsq(explainers(g, per_trip, total=True), (g["home_points"] + g["away_points"]).to_numpy() - (dh + da))
    return {"deserved": deserved, "per_trip": per_trip, "margin": margin, "total": total, "games": g.height}


fits = {}
for league, g in games_by_league.items():
    for season in sorted(set(picks[league]["season"].to_list())):
        past = g.filter((pl.col("season") == season - 1) & pl.col("has_stats"))
        if past.height < 200:
            print(f"{league} {season}: not enough {season - 1} games with play-by-play to learn from, so {season} games can't be reviewed.")
            continue
        f = fits[(league, season)] = fit_season(past)
        example = float(np.array([70 * 0.45, 70, 0.45, 1.0]) @ f["deserved"])
        print(f"{league}, learned from {season - 1} ({f['games']} games), used for {season} reviews:")
        print(f"  70 plays with 45% of them successful usually goes with about {example:.0f} points.")
        print(f"  Each turnover swings the margin by about {f['margin'][0]:.1f} points, each defensive or return touchdown by "
              f"{f['margin'][1]:.1f}, and each extra 20+ yard play by {f['margin'][2]:.1f}.")
        print(f"  Teams score about {f['per_trip']:.1f} points per trip inside the 40, and each point above that adds about "
              f"{f['margin'][3]:.2f} to the margin beyond what success rate already counts.")

## Step 4: Review every game

The **miss** is how much better the home team did than the model expected: the real margin minus the model's spread. It's split into parts that add up to the whole miss:

| Part | What it means |
|---|---|
| `efficiency` | How well the teams moved the ball: the margin their success rates usually mean, compared with the model's spread. This is the part where the teams simply played better or worse than the model expected |
| `turnovers` | The turnover margin |
| `return_tds` | Touchdowns on defense or special teams |
| `big_plays` | 20+ yard plays, beyond what success rate already counts |
| `finishing` | Points per trip inside the 40 compared with average, like scoring touchdowns instead of kicking field goals |
| `other` | Everything these stats don't measure: field position, kicking, penalties, late-game situations |

A miss of less than 7 points counts as **close**. For bigger misses, the main reason is the part (other than `other`) that pushed hardest in the direction of the miss, and the summary names up to two. Totals get the same treatment.

**Turnovers and return touchdowns are labeled luck.** In testing on 2020–2025, a team's turnover and return-touchdown luck in one game didn't carry over to its next game, in either league.

The review file has one row per game: the picks, the final score, the results against the spread and on the total, every part of the miss, both teams' stats, and the two summaries.

In [ ]:
CLOSE = 7.0       # within a touchdown counts as close
VERDICTS = {"close": "Close", "efficiency": "Teams played differently than expected", "turnovers": "Mostly luck: turnovers",
            "return_tds": "Mostly luck: defensive or return TDs", "big_plays": "Big plays", "finishing": "Finishing drives",
            "other": "Nothing in the stats stands out"}


def times(n, word):
    n = int(round(n))
    return f"{n} {word}" + ("" if n == 1 else "s")


def there_were(n):
    n = int(round(n))
    return f"There {'was' if n == 1 else 'were'} {n}"


def side_result(model_line, market_line, actual):
    """Win, loss or push for the side the model liked compared with the market; None if there was no market line."""
    if market_line is None or model_line is None or np.isnan(market_line):
        return None
    if model_line == market_line:
        return "no pick"
    if actual == market_line:
        return "push"
    return "win" if (model_line > market_line) == (actual > market_line) else "loss"


NAMED = PARTS[:5]          # everything except "other", which is whatever these stats don't explain
REASON_MIN, REST_MIN = 4.0, 8.0     # a part needs 4+ points to count as a reason; 8+ unexplained points get a mention


def reasons_for(r, prefix, miss):
    """The named parts that pushed 4+ points in the direction of the miss, biggest first."""
    pushes = {p: r[f"{prefix}{p}"] * np.sign(miss) for p in NAMED}
    return [p for p in sorted(pushes, key=pushes.get, reverse=True) if pushes[p] >= REASON_MIN][:2]


def main_reason(r, prefix, miss):
    if abs(miss) < CLOSE:
        return "close"
    found = reasons_for(r, prefix, miss)
    return found[0] if found else "other"


def explain_text(found, phrase, rest):
    if not found:
        return " Main reason: nothing in these stats stands out, so it came down to things like field position, kicking, penalties or late-game situations."
    text = (" Main reason: " if len(found) == 1 else " Biggest reasons: ") + "; and ".join(phrase[p] for p in found) + "."
    if rest >= REST_MIN:
        text += f" Another {rest:.0f} points don't show up in these stats: things like field position, kicking, penalties or late-game situations."
    return text


def margin_text(r):
    home, away, hp, ap, ms, miss = r["home_team"], r["away_team"], int(r["home_points"]), int(r["away_points"]), r["model_spread"], r["spread_miss"]
    if hp != ap:
        text = f"{home if hp > ap else away} beat {away if hp > ap else home} {max(hp, ap)}–{min(hp, ap)}."
    else:
        text = f"{home} and {away} tied {hp}–{ap}."
    pick = f"had {home if ms > 0 else away} by {abs(ms):.1f}" if abs(ms) >= 0.05 else "had it even"
    text += f" The model {pick}, so it was {abs(miss):.1f} points off."
    if r["reason"] == "close":
        return text + " Close: within a touchdown."
    b, w = ("h", "a") if miss > 0 else ("a", "h")          # b did better than the model expected
    B, W = (home, away) if miss > 0 else (away, home)
    s = lambda side, c: r[f"{side}_{c}"]
    size = lambda p: f"worth about {abs(r['part_' + p]):.0f} points"
    dm = r["deserved_margin"]
    usual = f"{home if dm > 0 else away} by {abs(dm):.0f}" if abs(dm) >= 0.5 else "an even game"
    phrase = {
        "efficiency": f"efficiency, {size('efficiency')}. {B} succeeded on {s(b, 'successes') / s(b, 'plays'):.0%} of its plays and {W} on "
                      f"{s(w, 'successes') / s(w, 'plays'):.0%}, which usually means {usual}",
        "turnovers": f"turnovers, {size('turnovers')}. {W} had {times(s(w, 'turnovers'), 'turnover')} and {B} {int(s(b, 'turnovers'))}",
        "return_tds": f"defensive and return touchdowns, {size('return_tds')}. {B} scored {int(s(b, 'return_tds'))} on defense or special teams"
                      + (f" and {W} {int(s(w, 'return_tds'))}" if s(w, "return_tds") else ""),
        "big_plays": f"big plays, {size('big_plays')}. {B} had {times(s(b, 'explosive'), 'play')} of 20+ yards and {W} {int(s(w, 'explosive'))}",
        "finishing": f"finishing drives, {size('finishing')}. {B} scored {int(s(b, 'trip_points'))} points on {times(s(b, 'trips'), 'trip')} "
                     f"inside the 40, and {W} {int(s(w, 'trip_points'))} on {int(s(w, 'trips'))}",
    }
    text += f" {B} did {abs(miss):.0f} points better than expected."
    return text + explain_text(reasons_for(r, "part_", miss), phrase, r["part_other"] * np.sign(miss))


def total_text(r):
    tm, mt = r["total_miss"], r["model_total"]
    text = f"Total points: {int(r['actual_total'])}, against the model's {mt:.1f}"
    text += f" and the market's {r['market_total']:g}" if r["market_total"] is not None else ""
    text += f", so {abs(tm):.1f} {'more' if tm > 0 else 'fewer'} than the model expected."
    if r["total_reason"] == "close":
        return text + " Close: within a touchdown."
    size = lambda p: f"worth about {abs(r['total_part_' + p]):.0f} points"
    phrase = {
        "efficiency": f"efficiency, {size('efficiency')}. The offenses succeeded on {r['h_successes'] / r['h_plays']:.0%} and "
                      f"{r['a_successes'] / r['a_plays']:.0%} of their plays, which usually means about {r['deserved_total']:.0f} points",
        "turnovers": f"turnovers, {size('turnovers')}. {there_were(r['h_turnovers'] + r['a_turnovers'])}",
        "return_tds": f"defensive and return touchdowns, {size('return_tds')}. {there_were(r['h_return_tds'] + r['a_return_tds'])}",
        "big_plays": f"big plays, {size('big_plays')}. The teams had {times(r['h_explosive'] + r['a_explosive'], 'play')} of 20+ yards",
        "finishing": f"finishing drives, {size('finishing')}. The teams scored {int(r['h_trip_points'] + r['a_trip_points'])} points on "
                     f"{times(r['h_trips'] + r['a_trips'], 'trip')} inside the 40",
    }
    return text + explain_text(reasons_for(r, "total_part_", tm), phrase, r["total_part_other"] * np.sign(tm))


def review(league):
    """One row per finished game the models picked, with stats, the size of each part of the miss, and plain-language summaries."""
    out = []
    joined = picks[league].join(games_by_league[league].filter("has_stats").drop("season", "week"), on="game_id", how="inner")
    for season in sorted(set(joined["season"].to_list())):
        if (league, season) not in fits:
            continue
        f, g = fits[(league, season)], joined.filter(pl.col("season") == season)
        dh, da = deserved_points(g, f["deserved"], "h"), deserved_points(g, f["deserved"], "a")
        margin_parts = explainers(g, f["per_trip"]) * f["margin"]
        total_parts = explainers(g, f["per_trip"], total=True) * f["total"]
        g = g.with_columns(
            actual_margin=pl.col("home_points") - pl.col("away_points"), actual_total=pl.col("home_points") + pl.col("away_points"),
            deserved_margin=pl.Series(dh - da), deserved_total=pl.Series(dh + da),
        ).with_columns(
            spread_miss=pl.col("actual_margin") - pl.col("model_spread"),       # positive = the home team did better than the model expected
            total_miss=pl.col("actual_total") - pl.col("model_total"),          # positive = more points than the model expected
            part_efficiency=pl.col("deserved_margin") - pl.col("model_spread"),
            total_part_efficiency=pl.col("deserved_total") - pl.col("model_total"),
            **{f"part_{p}": pl.Series(margin_parts[:, i]) for i, p in enumerate(PARTS[1:5])},
            **{f"total_part_{p}": pl.Series(total_parts[:, i]) for i, p in enumerate(PARTS[1:5])},
        ).with_columns(
            part_other=pl.col("spread_miss") - pl.sum_horizontal([f"part_{p}" for p in PARTS[:5]]),
            total_part_other=pl.col("total_miss") - pl.sum_horizontal([f"total_part_{p}" for p in PARTS[:5]]),
        )
        out.append(g)
    if not out:
        return None
    g = pl.concat(out)
    rows = []
    for r in g.iter_rows(named=True):
        extra = {"winner_right": None if r["actual_margin"] == 0 or r["model_spread"] is None else bool((r["model_spread"] > 0) == (r["actual_margin"] > 0)),
                 "ats": side_result(r["model_spread"], r["market_spread"], r["actual_margin"]),
                 "over_under": side_result(r["model_total"], r["market_total"], r["actual_total"])}
        if r["model_spread"] is not None:
            r["reason"] = main_reason(r, "part_", r["spread_miss"])
            extra.update(reason=r["reason"], verdict=VERDICTS[r["reason"]], summary=margin_text(r))
        if r["model_total"] is not None:
            r["total_reason"] = main_reason(r, "total_part_", r["total_miss"])
            extra.update(total_reason=r["total_reason"], total_summary=total_text(r))
        rows.append({"game_id": r["game_id"], **extra})
    text = pl.DataFrame(rows, schema={"game_id": g["game_id"].dtype, "winner_right": pl.Boolean, "ats": pl.String, "over_under": pl.String,
                                      "reason": pl.String, "verdict": pl.String, "summary": pl.String,
                                      "total_reason": pl.String, "total_summary": pl.String})
    g = g.join(text, on="game_id", how="left")
    stats = [f"{side}_{c}" for side in ("h", "a") for c in STATS]
    return g.select(
        pl.lit(league).alias("league"), "season", "week", "game_id", "kickoff_et", "away_team", "home_team", "away_points", "home_points",
        "margin_model", "model_spread", "market_spread", "model_p_home", "market_p_home", "actual_margin", "spread_miss",
        "winner_right", "ats", "reason", "verdict", "summary",
        "totals_model", "model_total", "market_total", "actual_total", "total_miss", "over_under", "total_reason", "total_summary",
        "deserved_margin", "deserved_total", *[f"part_{p}" for p in PARTS], *[f"total_part_{p}" for p in PARTS],
        *[pl.col(c).alias(c.replace("h_", "home_", 1) if c.startswith("h_") else c.replace("a_", "away_", 1)) for c in stats],
    ).sort("kickoff_et", "game_id")


reviews = {}
for league in picks:
    r = review(league) if league in games_by_league else None
    finished = picks[league].join(games_by_league[league], on="game_id", how="inner") if league in games_by_league else picks[league].head(0)
    waiting = finished.filter(~pl.col("has_stats")).height
    if r is None or r.is_empty():
        print(f"{league}: no finished games to review yet" + (f" ({waiting} finished, waiting for play-by-play)." if waiting else "."))
        continue
    reviews[league] = r
    round_cols = [c for c, t in r.schema.items() if t == pl.Float64 and c not in ("model_spread", "market_spread", "model_total", "market_total")]
    r.with_columns(pl.col(round_cols).round(2)).write_csv(REVIEW_FILES[league])
    print(f"{league}: reviewed {r.height} games and saved them to {REVIEW_FILES[league]}."
          + (f" {waiting} more finished games are waiting for play-by-play; the next run will add them." if waiting else ""))

## Step 5: The latest week

This shows the latest week with most of its games reviewed: every miss of a touchdown or more with its summary, the biggest totals misses, and a table of every game.

In [ ]:
for league, r in reviews.items():
    # The latest week with most of its picked games reviewed (midweek, the newest week may have only a game or two)
    picked = picks[league].group_by("season", "week").agg(picked=pl.len())
    weeks = r.group_by("season", "week").agg(reviewed=pl.len(), last=pl.col("kickoff_et").max()).join(picked, on=["season", "week"]).sort("last")
    full = weeks.filter(pl.col("reviewed") >= 0.5 * pl.col("picked"))
    w = (full if full.height else weeks).row(-1, named=True)
    week = r.filter((pl.col("season") == w["season"]) & (pl.col("week") == w["week"]))
    note = f" ({w['picked'] - w['reviewed']} more picked games still to come)" if w["reviewed"] < w["picked"] else ""
    print(f"===== {league}, {w['season']} week {w['week']}: {week.height} games reviewed{note} =====\n")
    spreads = week.filter(pl.col("reason").is_not_null())
    misses = spreads.filter(pl.col("reason") != "close").sort(pl.col("spread_miss").abs(), descending=True)
    print(f"Spreads: {spreads.height - misses.height} of {spreads.height} games were within a touchdown of the model."
          + (" The misses, biggest first:\n" if misses.height else "\n"))
    for g in misses.iter_rows(named=True):
        print(f"• {g['away_team']} at {g['home_team']}: {g['summary']}\n")
    big_totals = week.filter(pl.col("total_reason").is_not_null() & (pl.col("total_reason") != "close")).sort(pl.col("total_miss").abs(), descending=True).head(5)
    if big_totals.height:
        print("Biggest totals misses:\n")
        for g in big_totals.iter_rows(named=True):
            print(f"• {g['away_team']} at {g['home_team']}: {g['total_summary']}\n")
    print("Every game that week (miss = how much better the home team did than the model expected):")
    with pl.Config(tbl_rows=-1, fmt_str_lengths=40):
        display(week.sort(pl.col("spread_miss").abs(), descending=True, nulls_last=True).select(
            "away_team", "home_team", score=pl.format("{}-{}", pl.col("away_points").cast(pl.Int32), pl.col("home_points").cast(pl.Int32)),
            model_spread=pl.col("model_spread").round(1), market_spread="market_spread", miss=pl.col("spread_miss").round(1),
            ats="ats", verdict="verdict", total_miss=pl.col("total_miss").round(1), over_under="over_under"))

## Step 6: The big picture

Across every reviewed game: how often the model picked the winner, its average miss next to the market's on the same games, its records against the spread and on totals, and why it missed when it missed by a touchdown or more.

For comparison, on the 2025 season the main reason for a miss of a touchdown or more was efficiency about 40–50% of the time. Turnovers and finishing drives were each the main reason for about 15–25%, and nothing in the stats stood out for about 10%.

In [ ]:
def show(value, pattern):
    return "–" if value is None else pattern.format(value)


def record(df, col):
    s = df[col].drop_nulls()
    wins, losses, pushes = (s == "win").sum(), (s == "loss").sum(), (s == "push").sum()
    rate = f"  ({wins / (wins + losses):.1%})" if wins + losses else ""
    return f"{wins}-{losses}-{pushes}{rate}"


for league, r in reviews.items():
    print(f"===== {league}: every reviewed game =====")
    m = r.filter(pl.col("model_spread").is_not_null())
    lined = m.filter(pl.col("market_spread").is_not_null())
    totals = r.filter(pl.col("model_total").is_not_null())
    totals_lined = totals.filter(pl.col("market_total").is_not_null())
    display(pl.DataFrame([
        {"measure": "Games reviewed", "value": f"{r.height}"},
        {"measure": "Picked the winner", "value": show(m["winner_right"].drop_nulls().mean(), "{:.1%}")},
        {"measure": "Spread miss: model", "value": show(m["spread_miss"].abs().mean(), "{:.1f} points")},
        {"measure": "Spread miss: market, same games", "value": show((lined["actual_margin"] - lined["market_spread"]).abs().mean(), "{:.1f} points")
                                                              + (f" (model {lined['spread_miss'].abs().mean():.1f} on these)" if 0 < lined.height < m.height else "")},
        {"measure": "Against the spread (break-even 52.4%)", "value": record(m, "ats")},
        {"measure": "Totals miss: model", "value": show(totals["total_miss"].abs().mean(), "{:.1f} points")},
        {"measure": "Totals miss: market, same games", "value": show((totals_lined["actual_total"] - totals_lined["market_total"]).abs().mean(), "{:.1f} points")},
        {"measure": "Over/unders (break-even 52.4%)", "value": record(totals, "over_under")},
    ]))
    big = m.filter(pl.col("reason") != "close")
    print(f"Why the model missed by a touchdown or more ({big.height} of {m.height} games):")
    display(big.group_by("verdict").agg(games=pl.len()).with_columns(share=(pl.col("games") / max(big.height, 1)).round(3)).sort("games", descending=True))
    print("Average size of each part of the miss, in points, ignoring direction:")
    display(m.select(
        pl.len().alias("games"),
        *[pl.col(f"part_{p}").abs().mean().round(1).alias(p) for p in PARTS],
        pl.col("spread_miss").abs().mean().round(1).alias("whole_miss"),
    ))

## Step 7: Teams that keep surprising the model

For each team this season: how many points better or worse than the model expected it has done on average (`beat_model_by`), and how much of that came from better play (`from_play`) or from luck (`from_luck`, meaning turnovers and return touchdowns). The parts don't add up to the total, because finishing, big plays and everything else are left out.

Read this as what happened, not what will happen. In testing on 2020–2025:
- **`beat_model_by` didn't carry over** in either league. Teams that beat the model for several weeks didn't keep beating it the next week: a lot of it is luck, and the ratings catch up.
- **`from_play` carried over a little in the NFL.** Teams that played 5+ points better than expected on average over the season so far kept beating the model by about 1.5 points the next week, in 5 of the 6 seasons. College showed no such pattern. That hints the NFL model adjusts too slowly, and it points to a fix worth testing: unlike the college model, the NFL margin model has no points or efficiency ratings.

In [ ]:
MIN_GAMES = 3

for league, r in reviews.items():
    season = r["season"].max()
    m = r.filter((pl.col("season") == season) & pl.col("model_spread").is_not_null())
    # Each game twice, once from each team's side: positive = the team did better than the model expected
    sides = pl.concat([
        m.select(team="home_team", miss="spread_miss", efficiency="part_efficiency", luck=pl.col("part_turnovers") + pl.col("part_return_tds")),
        m.select(team="away_team", miss=-pl.col("spread_miss"), efficiency=-pl.col("part_efficiency"),
                 luck=-(pl.col("part_turnovers") + pl.col("part_return_tds"))),
    ])
    teams = sides.group_by("team").agg(
        games=pl.len(),
        beat_model_by=pl.col("miss").mean().round(1),
        from_play=pl.col("efficiency").mean().round(1),
        from_luck=pl.col("luck").mean().round(1),
    ).filter(pl.col("games") >= MIN_GAMES)
    print(f"===== {league}, {season}: teams with {MIN_GAMES}+ reviewed games =====")
    if teams.is_empty():
        print(f"No team has {MIN_GAMES} reviewed games yet. This fills in as the weeks go by.\n")
        continue
    print("Beat the model by the most, on average (from_play = better play than expected, from_luck = turnovers and return TDs):")
    display(teams.sort("beat_model_by", descending=True).head(8))
    print("Fell short of the model by the most:")
    display(teams.sort("beat_model_by").head(8))

## Next steps

1. **Run this every Tuesday**, after the model notebooks.
2. **The app.** The review files are ready for a "how did we do" screen, and for an AI to explain each game from the facts in them.
3. **NFL efficiency ratings.** The review hints that the NFL margin model adjusts too slowly, so the next model test is adding success-rate ratings to it, like the college model's.